In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

from src import quality

print(sklearn.__version__, np.__version__)

1.9.1 2.5.4


In [2]:
docs = pd.Series(["Apple shares rise", "Apple shares fall", "Oil prices fall"])
vec = TfidfVectorizer()
X = vec.fit_transform(docs)
print(type(X).__name__, X.shape)
print(vec.get_feature_names_out())

csr_matrix (3, 6)
['apple' 'fall' 'oil' 'prices' 'rise' 'shares']


In [3]:
dense = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
print(dense.round(3))
print(vec.idf_.round(3))
print(np.linalg.norm(X.toarray(), axis=1))

   apple   fall    oil  prices   rise  shares
0  0.518  0.000  0.000   0.000  0.681   0.518
1  0.577  0.577  0.000   0.000  0.000   0.577
2  0.000  0.474  0.623   0.623  0.000   0.000
[1.288 1.288 1.693 1.693 1.693 1.288]
[1. 1. 1.]


In [4]:
analyzer = vec.build_analyzer()
print(analyzer("$APPL up 5% to $230, a record"))

['appl', 'up', 'to', '230', 'record']


In [5]:
raw = pd.Series([
    "Apple shares rise https://t.co/x81kQp",
    "Apple shares rise https://t.co/Zr4mB2",
    "Oil prices fall https://t.co/Qe7Lw0",
])

for name, s in [("raw", raw), ("normalized", quality.normalize_text(raw))]:
    v = TfidfVectorizer()
    M = v.fit_transform(s)
    print(name, v.get_feature_names_out())
    print("   cos(0, 1) = ", round(float((M[0] @ M[1].T).toarray()[0, 0]), 3))

raw ['apple' 'co' 'fall' 'https' 'oil' 'prices' 'qe7lw0' 'rise' 'shares'
 'x81kqp' 'zr4mb2']
   cos(0, 1) =  0.709
normalized ['apple' 'fall' 'oil' 'prices' 'rise' 'shares']
   cos(0, 1) =  1.0


In [6]:
S = (X @ X.T).toarray()
print(S.round(3))

[[1.    0.598 0.   ]
 [0.598 1.    0.273]
 [0.    0.273 1.   ]]


In [7]:
nn = NearestNeighbors(n_neighbors=2, metric="cosine").fit(X)
dist, idx = nn.kneighbors(X)
print(dist.round(3))
print(idx)

[[0.    0.402]
 [0.    0.402]
 [0.    0.727]]
[[0 1]
 [1 0]
 [2 1]]


In [8]:
twins = pd.Series(["Shares rise today", "Shares rise today!", "Profit falls"])
keys = quality.normalize_text(twins)
print(keys.tolist())
T = TfidfVectorizer().fit_transform(keys)
dist_t, idx_t = NearestNeighbors(n_neighbors=2, metric="cosine").fit(T).kneighbors(T)
print(T)

['shares rise today', 'shares rise today!', 'profit falls']
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 8 stored elements and shape (3, 5)>
  Coords	Values
  (0, 3)	0.5773502691896257
  (0, 2)	0.5773502691896257
  (0, 4)	0.5773502691896257
  (1, 3)	0.5773502691896257
  (1, 2)	0.5773502691896257
  (1, 4)	0.5773502691896257
  (2, 1)	0.7071067811865476
  (2, 0)	0.7071067811865476


In [9]:
rows = np.arange(len(idx_t))
print(rows)
print(idx_t[:, 0] == rows)
pick = np.where(idx_t[:, 0] == rows, 1, 0)
print(pick)
print(idx_t[rows, pick])
print(idx_t[:, pick])

[0 1 2]
[ True False  True]
[1 0 1]
[1 0 1]
[[1 0 1]
 [1 0 1]
 [1 2 1]]


In [12]:
ref = pd.Series(["Apple shares rise after earnings",
                 "Oil prices fall on weak demand",
                 "Bank names new chief executive"])
qry = pd.Series(["Apple shares rise after strong earnings", "Gold hits record high"])

nn_q = quality.nearest_neighbors(ref, qry)
print(nn_q)
assert list(nn_q.columns) == ["query_pos", "match_pos", "similarity"]
assert nn_q.loc[0, "match_pos"] == 0 and 0.85 < nn_q.loc[0, "similarity"] < 0.9
assert nn_q.loc[1, "similarity"] < 1e-9            # no shared words

nn_s = quality.nearest_neighbors(twins)            # self mode, the trap from cell 7
print(nn_s)
assert (nn_s["match_pos"] != nn_s["query_pos"]).all()   # never matched with itself
assert list(nn_s["match_pos"][:2]) == [1, 0]            # the twins find each other
assert np.isclose(nn_s.loc[0, "similarity"], 1)

shifted = qry.set_axis([9543, 9544])               # index labels like valid's
assert quality.nearest_neighbors(ref, shifted).equals(nn_q)   # positions, not labels
print("nearest_neighbors OK")

   query_pos  match_pos  similarity
0          0          0    0.874619
1          1          0    0.000000
   query_pos  match_pos  similarity
0          0          1         1.0
1          1          0         1.0
2          2          1         0.0
nearest_neighbors OK
